In [21]:
!pip install -q ultralytics

In [22]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

In [23]:
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
ret,frame = cap.read()
ret2, frame2 = cap.read()

In [24]:
def get_centroids(frame):
  result = model.predict(frame)
  boxes = result[0].boxes.xyxy.cpu().numpy()
  conf = result[0].boxes.conf.cpu().numpy()
  centroid=[]

  for box,c in zip(boxes,conf):
    if c < 0.5:
      continue
    x1,y1,x2,y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2
    centroid.append((cx,cy))

  return centroid

In [25]:
centroid_frame1 = get_centroids(frame)
centroid_frame2 = get_centroids(frame2)


0: 384x640 11 persons, 133.4ms
Speed: 5.4ms preprocess, 133.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 128.9ms
Speed: 5.5ms preprocess, 128.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)


In [26]:
players_position = {}
next_id = 0
for c in centroid_frame1:
  players_position[next_id] = [c]
  next_id +=1
print(players_position)

{0: [(np.float32(403.42545), np.float32(304.4803))], 1: [(np.float32(99.70708), np.float32(275.87146))], 2: [(np.float32(589.9822), np.float32(353.16367))], 3: [(np.float32(711.9473), np.float32(237.68384))], 4: [(np.float32(146.1174), np.float32(227.2714))], 5: [(np.float32(555.9518), np.float32(242.25488))], 6: [(np.float32(593.5323), np.float32(291.34247))], 7: [(np.float32(403.35294), np.float32(258.49933))]}


In [27]:
print(fps)

25.0


In [28]:
import math
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")   # reopen from frame 0 — the old cap object is exhausted
frame_count = 0
max_frames = 50   # quick test limit — remove once logic is confirmed correct
max_distance = 50
max_speed = 35

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    if frame_count > max_frames:
        break

    centroid = get_centroids(frame)
    used_ids = set()
    for c in centroid:
        best_id = None
        best_distance = float("inf")
        for pid, pos in players_position.items():
          d = math.dist(c, pos[-1])
          if d<best_distance and pid not in used_ids:
            best_id = pid
            best_distance = d
        if best_distance < max_distance:
          speed = best_distance * fps
          if speed > max_speed:
              pass  # <-- what happens here? (hint: nothing gets appended, no new id either)
          else:
              players_position[best_id].append(c)
              used_ids.add(best_id)
        else:
            players_position[next_id] = [c]
            next_id += 1

print(players_position)
print(next_id)


0: 384x640 11 persons, 120.0ms
Speed: 4.8ms preprocess, 120.0ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 126.2ms
Speed: 4.4ms preprocess, 126.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 140.8ms
Speed: 5.2ms preprocess, 140.8ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 125.8ms
Speed: 5.2ms preprocess, 125.8ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 124.4ms
Speed: 5.3ms preprocess, 124.4ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 127.6ms
Speed: 4.9ms preprocess, 127.6ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 123.9ms
Speed: 4.5ms preprocess, 123.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 125.0ms
Speed: 5.4ms preprocess, 125.0ms inference, 1.2ms postproc

In [29]:
players_position = {pid: pos for pid, pos in players_position.items() if len(pos) >= 5}

In [30]:
player_distances = {}

for pid, history in players_position.items():
    total = 0
    for i in range(len(history) - 1):
        x1, y1 = history[i]
        x2, y2 = history[i+1]
        d = ((x2-x1)**2 + (y2-y1)**2)**0.5
        total += d
    player_distances[pid] = total

print(player_distances)

{0: np.float32(6.95741), 1: np.float32(11.480691), 2: np.float32(7.782707), 3: np.float32(7.6705875), 4: np.float32(6.658999), 5: np.float32(17.154175), 6: np.float32(8.01182), 7: np.float32(5.366406), 8: np.float32(8.158396)}


In [31]:
def get_jersey_crop(frame, box):
    # cast to int since slicing needs whole numbers, not the floats YOLO gives
    x1, y1, x2, y2 = map(int, box)

    height = y2 - y1
    # only take top 35% of box height to isolate jersey, skip shorts/legs
    new_y2 = int(y1 + (0.35 * height))

    # rows (y) first, then columns (x) — standard image slicing order
    return frame[y1:new_y2, x1:x2]

In [32]:

def get_avg_color(crop):
  return crop.mean(axis = (0,1))

In [33]:
avg_colors = []
result = model.predict(frame)
boxes = result[0].boxes.xyxy.cpu().numpy()
for box in boxes:
  crop = get_jersey_crop(frame, box)
  avg_color = get_avg_color(crop)
  avg_colors.append(avg_color)

print(len(avg_colors))
print(avg_colors[0])


0: 384x640 12 persons, 122.6ms
Speed: 4.2ms preprocess, 122.6ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
12
[     75.588      135.89      121.67]


In [34]:
data = np.array(avg_colors, dtype=np.float32)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
compactness, labels, centers = cv2.kmeans(data, 2, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

print(compactness, labels, centers)

4211.715896606445 [[1]
 [0]
 [0]
 [0]
 [0]
 [1]
 [1]
 [1]
 [0]
 [0]
 [1]
 [1]] [[     109.78      159.15      142.31]
 [     83.438      120.89      123.11]]


In [35]:
player_team = {}
for i,box in enumerate(boxes):
  x1, y1, x2, y2 = box
  cx = (x1+x2)/2
  cy= (y1+y2)/2

  best_id = None
  best_distance = float("inf")
  for pid,pos in players_position.items():
    d = math.dist((cx,cy), pos[-1])
    if d<best_distance:
      best_distance = d
      best_id = pid

  if best_id not in player_team :
    player_team[best_id] = labels[i]

print(player_team)

{4: array([1], dtype=int32), 2: array([0], dtype=int32), 7: array([0], dtype=int32), 5: array([0], dtype=int32), 6: array([0], dtype=int32), 1: array([1], dtype=int32), 3: array([0], dtype=int32)}


In [36]:
team_distance = {}
for pid, dist in player_distances.items():
  if pid not in player_team:
    continue
  team = int(player_team[pid])
  if team not in team_distance:
    team_distance[team]= 0
  team_distance[team] += dist

print(team_distance)


{1: np.float32(18.13969), 0: np.float32(45.98569)}


/tmp/ipykernel_1171/3581967056.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  team = int(player_team[pid])


In [37]:
import math
def get_player_speeds(position_history,fps):
  speeds =[]
  for i in range(1,len(position_history)):
    prev_point = position_history[i-1]
    current_point = position_history[i]
    x1, y1 = prev_point
    x2, y2 = current_point
    distance = math.sqrt((x2-x1)**2 + (y2-y1)**2)
    speed = distance*fps
    speeds.append(speed)
  return speeds



fps = cap.get(cv2.CAP_PROP_FPS)

# Pass a valid player ID (e.g., 0)
speeds = get_player_speeds(players_position[0], fps)
print(speeds)

[0.0, 1.2132574937893303, 5.197432280946672, 1.9628813076747083, 6.400388152488996, 7.38872667111299, 1.915143070479822, 4.116378008120604, 2.1572977482131543, 5.465851168726877, 5.012995494844548, 2.6597345248250246, 4.414651428449144, 6.4130978648802515, 6.34969825177869, 4.798942771818355, 12.685095270236655, 9.441504429412074, 19.2487775179118, 29.685121993141415, 14.013213306327579, 23.39506280982743]


In [38]:
def count_sprints(speeds, threshold):

  sprint_count = 0
  was_sprinting = False
  for i in speeds:
    is_sprinting = i > threshold
    if is_sprinting and not was_sprinting :
      sprint_count +=1
    was_sprinting = is_sprinting

  return sprint_count

counts = count_sprints(speeds, 30)
print(counts)

0


In [39]:
def build_player_summary(players_position, player_team, player_distances, fps, sprint_threshold):
  player_summary = {}
  for player_id,pos_history in players_position.items():
    # loop over each tracked player to pull together their stats into one summary
    team = player_team.get(player_id, "unknown")
    if not isinstance(team, str):
      team = team.item()
    distance = player_distances[player_id]

    speed = get_player_speeds(pos_history,fps)
    sprint_count = count_sprints(speed,sprint_threshold)

    player_summary[player_id] = {
        "team" : team,
        "distance" : distance,
        "speed" : speed,
        "sprint_count" : sprint_count
    }

  return player_summary


build_player_summary(players_position, player_team, player_distances, fps, 30)


{0: {'team': 'unknown',
  'distance': np.float32(6.95741),
  'speed': [0.0,
   1.2132574937893303,
   5.197432280946672,
   1.9628813076747083,
   6.400388152488996,
   7.38872667111299,
   1.915143070479822,
   4.116378008120604,
   2.1572977482131543,
   5.465851168726877,
   5.012995494844548,
   2.6597345248250246,
   4.414651428449144,
   6.4130978648802515,
   6.34969825177869,
   4.798942771818355,
   12.685095270236655,
   9.441504429412074,
   19.2487775179118,
   29.685121993141415,
   14.013213306327579,
   23.39506280982743],
  'sprint_count': 0},
 1: {'team': 1,
  'distance': np.float32(11.480691),
  'speed': [0.0,
   8.254440203689546,
   14.151035528447245,
   10.653396729259832,
   9.078945127790794,
   6.858209703258357,
   13.512477839096205,
   25.213981335914674,
   20.44929784925259,
   33.71313638245615,
   20.899000784154552,
   23.28587761273388,
   32.13667500266773,
   14.870531617129815,
   15.077419412736262,
   12.872851285054839,
   3.3909983577712697,
   